In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import sys
sys.path.append("../../../../LoRO")

from utils import *

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("valhalla/bart-large-finetuned-squadv1")
model = AutoModelForQuestionAnswering.from_pretrained("valhalla/bart-large-finetuned-squadv1")

You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1'}. The number of labels wil be overwritten to 2.
/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'LABEL_0', '1': 'LABEL_1'}. The number of labels wil be overwritten to 2.

In [3]:
print(model)

BartForQuestionAnswering(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (f

In [4]:
#obfuscate model
model = model_obfuscation(model)
print(model)

Obfuscating: model.encoder.layers.0.self_attn.k_proj
Obfuscating: model.encoder.layers.0.self_attn.v_proj
Obfuscating: model.encoder.layers.0.self_attn.q_proj
Obfuscating: model.encoder.layers.0.self_attn.out_proj
Obfuscating: model.encoder.layers.0.fc1
Obfuscating: model.encoder.layers.0.fc2
Obfuscating: model.encoder.layers.1.self_attn.k_proj
Obfuscating: model.encoder.layers.1.self_attn.v_proj
Obfuscating: model.encoder.layers.1.self_attn.q_proj
Obfuscating: model.encoder.layers.1.self_attn.out_proj
Obfuscating: model.encoder.layers.1.fc1
Obfuscating: model.encoder.layers.1.fc2
Obfuscating: model.encoder.layers.2.self_attn.k_proj
Obfuscating: model.encoder.layers.2.self_attn.v_proj
Obfuscating: model.encoder.layers.2.self_attn.q_proj
Obfuscating: model.encoder.layers.2.self_attn.out_proj
Obfuscating: model.encoder.layers.2.fc1
Obfuscating: model.encoder.layers.2.fc2
Obfuscating: model.encoder.layers.3.self_attn.k_proj
Obfuscating: model.encoder.layers.3.self_attn.v_proj
Obfuscating:

In [5]:
model = obfus_inference_mode(model)

In [6]:
question_answerer = pipeline("question-answering", model=model, tokenizer=tokenizer, device='cuda')

In [7]:
context = 'Tom likes playing with Jerry.'
question = 'Who does Tom like playing with?'
result = question_answerer(question=question, context=context)
print(result)

{'score': 0.020408377051353455, 'start': 0, 'end': 3, 'answer': 'Tom'}


In [8]:
dataset = load_dataset("rajpurkar/squad")['validation']
print(dataset)

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10570
})


In [9]:
correct = 0
total = 0

for i in tqdm.tqdm(range(10570)):
    result = question_answerer(question=dataset[i]['question'], context=dataset[i]['context'])
    for answer in dataset[i]['answers']['text']: # exact match
        if answer == result['answer']:
            correct += 1
            break
    total += 1

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

100%|██████████| 10570/10570 [03:16<00:00, 53.85it/s]

correct:81, total:10570, accuracy:0.007663197729422895
